### Imports

In [1]:
import sys
import os

sys.path.append(os.path.abspath("..")) 

import json
import pandas as pd
import re
from datetime import datetime, timezone
from clean_text import clean_text
from data_class.raw_data import RawData
from tqdm.auto import tqdm

### Helper functions for cleaning

In [18]:
def normalize_verdict(text: str) -> str:
    """Normalize verdict by removing label prefixes, explanatory text, and trailing punctuation"""
    # Drop a leading "<label>:" (e.g., "Rating:", "Marka:", "MISLEADING:")
    text = re.sub(r'^\s*[^:]+:\s*', '', text, count=1)
    
    # Remove everything after the first sentence (ending with .)
    # This handles cases like "FALSE. Rappler agrees with..."
    verdict = text.split('.')[0]
    
    # Remove trailing punctuation (., :, etc.)
    verdict = verdict.rstrip('.:;,!?')
    
    # Remove leading/trailing whitespace
    verdict = verdict.strip()
    
    return verdict

def standardize_verdict(verdict: str) -> str:
    """Map verdicts to labels suitable for XLM-RoBERTa classification"""
    verdict = normalize_verdict(verdict)
    
    verdict_upper = verdict.upper()

    if any(term in verdict_upper for term in [
        'MISLEADING'
    ]):
        return "MISLEADING"

    if any(term in verdict_upper for term in [
        'ALTERED', 'MANIPULA','MANILUPADONG', 'MANIPULATED'
    ]):
        return "MANIPULATED"
    
    if any(term in verdict_upper for term in [
        'PARTLY FALSE', 'MIXED'
    ]):
        return "PARTLY-FALSE"
    
    if any(term in verdict_upper for term in [
        'FALSE', 'DIRI', 'DILI', 'HINDI', 'INDI', 'HOAX', "BAKONG", 
    ]):
        return "FALSE"
    

    if any(term in verdict_upper for term in [
        'MISSING', 'NEEDS CONTEXT', 'KULANG', 'CONTEXT', 'NO PROOF'
    ]):
        return "MISSING-CONTEXT"
    
    # SATIRE - clearly satirical (filter out or separate dataset)
    if 'SATIRE' in verdict_upper:
        return "SATIRE"
    
    raise Exception(f"Unable to classify Verdict: {verdict}")

def normalize_date(iso_str: str) -> str:
    """Convert +8 timezone to +0"""

    dt = datetime.fromisoformat(iso_str)
    normalized_date = dt.astimezone(timezone.utc)

    return normalized_date.isoformat()


def clean_article(article: RawData):
    """Clean a single article entry"""
    cleaned: RawData = article.copy()
    
    # Normalize date timezone
    cleaned['publish_date'] = normalize_date(cleaned['publish_date'])
    
    # Clean text fields
    for field in ['title', 'content', 'claim']:
        if field in cleaned and cleaned[field]:
            cleaned[field] = clean_text(cleaned[field])

    # Normalize verdict
    cleaned["verdict"] = standardize_verdict(cleaned["verdict"])
    #cleaned["verdict"] = cleaned["verdict"].upper()

    
    # Ensure authors is a list
    if 'authors' in cleaned and cleaned['authors'] is None:
        cleaned['authors'] = []

    # Add source bias from: https://mediabiasfactcheck.com
    cleaned["source_bias"] = "LEFT-CENTER"

    # Upper case necessary props
    cleaned["source"] = cleaned["source"].upper()
    cleaned["type"] = cleaned["type"].upper()
    
    return cleaned

### Load in dataset

In [19]:
with open('../outputs/rappler-factcheck.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

### Clean 

In [20]:
# Clean all articles
cleaned_data = [clean_article(article) for article in tqdm(data, desc="Cleaning articles")]

Cleaning articles:   0%|          | 0/3078 [00:00<?, ?it/s]

### Convert to DF and inspect

In [21]:
df = pd.DataFrame(cleaned_data)

print(f"\nSample of cleaned data:")
print(df[['title', 'content', 'claim', 'verdict', 'publish_date', 'source_bias']].head())


Sample of cleaned data:
                                               title  \
0  FACT CHECK: Link to Makati City ‘Pamaskong Han...   
1  FACT CHECK: Marcoleta is a senator, not the Ho...   
2  FACT CHECK: Registration links for ‘Christmas ...   
3  FACT CHECK: Duterte not found unconscious in I...   
4  FACT CHECK: Supreme Court did not file charges...   

                                             content  \
0  Claim: Makati City residents can sign up for t...   
1  Claim: Senator Rodante Marcoleta is the new Ho...   
2  Claim: Parents with children enrolled from the...   
3  Claim: Former president Rodrigo Duterte was fo...   
4  Claim: The Supreme Court﻿ (SC) has filed charg...   

                                               claim verdict  \
0  Makati City residents can sign up for the loca...   FALSE   
1  Senator Rodante Marcoleta is the new House spe...   FALSE   
2  Parents with children enrolled from the elemen...   FALSE   
3  Former president Rodrigo Duterte was found

### Inspect all verdict types

In [22]:
unique_verdicts = df['verdict'].unique()
print(f"\nUnique verdicts ({len(unique_verdicts)}):")
for verdict in sorted(unique_verdicts):
    count = (df['verdict'] == verdict).sum()
    print(f"  - {verdict}: {count} articles")


Unique verdicts (6):
  - FALSE: 2822 articles
  - MANIPULATED: 29 articles
  - MISLEADING: 35 articles
  - MISSING-CONTEXT: 146 articles
  - PARTLY-FALSE: 31 articles
  - SATIRE: 15 articles


### Save cleaned data to separate JSON files

In [23]:
output_dir = '../outputs_clean/rappler_factcheck'

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

with open(f'{output_dir}/rappler_factcheck_cleaned.json', 'w', encoding='utf-8') as f:
    json.dump(cleaned_data, f, indent=2, ensure_ascii=False)

print("Cleaning complete! Files saved.")

Cleaning complete! Files saved.
